# 05 — Mapas interactivos: ArcGIS datos + Folium visualización

# ArcGIS API for Python → consulta y obtiene datos geográficos reales
# Folium → los pinta en un mapa interactivo HTML
# Este es el enfoque profesional cuando no tienes ArcGIS Pro en tu máquina

In [11]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import folium

gis = GIS()
print('ArcGIS conectado — listo')
print(f'Folium version: {folium.__version__}')

ArcGIS conectado — listo
Folium version: 0.20.0


## 1. Mapa base centrado en España

In [14]:
# location = [latitud, longitud] — España centrada
mapa = folium.Map(location=[40.4, -3.7], zoom_start=6)
mapa

## 2. Datos reales de ArcGIS → pintados en el mapa

In [17]:
import math

def webmercator_a_wgs84(x, y):
    """Convierte coordenadas Web Mercator (metros) a WGS84 (grados lat/lon)."""
    lon = (x / 20037508.342) * 180
    lat = math.degrees(2 * math.atan(math.exp(y / 20037508.342 * math.pi)) - math.pi / 2)
    return lat, lon

URL_PARADAS = 'https://services1.arcgis.com/nCKYwcSONQTkPA4K/arcgis/rest/services/ParadasInterurbanosBarcelona/FeatureServer/0'
capa = FeatureLayer(URL_PARADAS)

paradas = capa.query(where='1=1', out_fields='*', result_record_count=50)
print(f'Paradas obtenidas de ArcGIS: {len(paradas.features)}')

mapa2 = folium.Map(location=[41.4, 2.1], zoom_start=9)

for i, feature in enumerate(paradas.features):
    a = feature.attributes
    geo = feature.geometry
    lat, lon = webmercator_a_wgs84(geo['x'], geo['y'])

    if i < 3:
        print(f"  [{i}] lat={lat:.4f}, lon={lon:.4f} — {a['Nombre_de_la_parada'][:40]}")

    folium.CircleMarker(
        location=[lat, lon],
        radius=7,
        color='steelblue',
        fill=True,
        fill_color='steelblue',
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{a['Nombre_de_la_parada']}</b><br>"
            f"Línea: {a['Nombre_de_la_linea']}<br>"
            f"Operador: {a['Operador']}",
            max_width=250
        )
    ).add_to(mapa2)

print('\nHaz clic en un punto azul para ver sus atributos.')
mapa2

Paradas obtenidas de ArcGIS: 50
  [0] lat=38.1402, lon=8.3272 — Casino Rabassada - av. Can Cortès (A)
  [1] lat=38.1541, lon=8.3121 — cruïlla Colònia Parès (D)
  [2] lat=38.1516, lon=8.3160 — ctra. Rabassada (Mas del Bosc) (A)

Haz clic en un punto azul para ver sus atributos.


## 3. Nuestro propio punto — simulación del pipeline YOLO

In [20]:
# Esto simula exactamente lo que hará el pipeline final:
# YOLO detecta un objeto → Python crea este punto → aparece en el mapa

deteccion_yolo = {
    'lat': 40.4654,
    'lon': -3.6892,
    'tipo': 'Señal de obra',
    'confianza': 94,
    'operario': 'Carlos',
    'fecha': '2026-06-05'
}

mapa3 = folium.Map(location=[deteccion_yolo['lat'], deteccion_yolo['lon']], zoom_start=14)

folium.Marker(
    location=[deteccion_yolo['lat'], deteccion_yolo['lon']],
    popup=folium.Popup(
        f"<b>{deteccion_yolo['tipo']}</b><br>"
        f"Confianza YOLO: {deteccion_yolo['confianza']}%<br>"
        f"Operario: {deteccion_yolo['operario']}<br>"
        f"Fecha: {deteccion_yolo['fecha']}",
        max_width=200
    ),
    icon=folium.Icon(color='red', icon='warning-sign')
).add_to(mapa3)

mapa3

## Guardar el mapa como HTML

In [ ]:
# Exportar a HTML — se puede abrir en cualquier navegador y compartir
mapa3.save('mapa_deteccion_yolo.html')
print('Mapa guardado como mapa_deteccion_yolo.html')
print('Abre ese archivo en tu navegador para verlo fuera de Jupyter.')

## Concepto clave

La celda 3 es el corazón del proyecto:
- **geometry** = coordenadas GPS de donde se hizo la foto
- **attributes** = lo que YOLO detectó + quién + cuándo

En el flujo real YOLO rellena ese diccionario automáticamente.
Folium (o ArcGIS) lo pinta en el mapa al instante.